In [2]:
import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import re 
import torch
import numpy as np

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [4]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    with open(file_path, "rb") as file:
        data = file.read()
    if jsonl:
        output = decoder.decode_lines(data)
    else:
        output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)

def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp



In [342]:
def get_para_id_context_set(json_list):
    para_id_set = set([x['para_id'] for x in json_list])
    context_set = set([x['context'].strip() for x in json_list])
    return para_id_set, context_set

def get_para_id_to_context_dict(json_list):
    para_id_to_context_dict = {}
    for i in json_list:
        para_id = i['para_id']
        context = i['context']
        # question_answer = i['question'] + i['answer']
        # url = i['url']
        raw_ocr = i['raw_ocr']
        publication_date = i['publication_date']
        if para_id_to_context_dict.get(para_id, None) is None:
            para_id_to_context_dict[para_id] = {
                "context": [context], 
                # "question_answer": [question_answer], 
                # 'url': url, 
                "raw_ocr": raw_ocr,
                "publication_date": publication_date,
            }
        else:
            para_id_to_context_dict[para_id]['context'].append(context)
            # para_id_to_context_dict[para_id]['question_answer'].append(question_answer)
            # para_id_to_context_dict[para_id]['url'].append(url)
    for k, v in para_id_to_context_dict.items():
        para_id_to_context_dict[k]['context'] = list(set(v['context']))
    return para_id_to_context_dict

# Read the corpus data

In [341]:
DATA_PATH = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/ChroniclingAmericaQA")

train = read_json(DATA_PATH / "train.json")
dev = read_json(DATA_PATH / "dev.json")
test = read_json(DATA_PATH / "test.json")

print(train[0])

The file is of type: <class 'list'>
The file contains 439302 items.
The file is of type: <class 'list'>
The file contains 24111 items.
The file is of type: <class 'list'>
The file contains 24084 items.
{'query_id': 'train_1', 'question': 'Who is the author of the book, "Horrors of Slavery, or the American Turf in Tripoli"?', 'answer': 'WILLIAM RAY', 'org_answer': 'WILLIAM RAY', 'para_id': 'New_Hampshire_18070804_1', 'context': "Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of Slavery, or the American Turf in Tripoli,” VOTARIES of Freedom, arm! The British Lion roars! Legions of Valor, take th’ alarm—; Rash, rush to guard our shores! Behold the horrid deed— Your brethren gasping lie! Beneath a tyrant’s hand they bleed— They groan—they faint—they die. Veterans of seventy-six, Awake the slumbering sword;— Hearts of your murderous foes transf

In [ ]:
# para_id_set_train, context_set_train = get_para_id_context_set(train)
# para_id_set_dev, context_set_dev = get_para_id_context_set(dev)
# para_id_set_test, context_set_test = get_para_id_context_set(test)

# print(len(para_id_set_train.union(para_id_set_dev).union(para_id_set_test)))
# print(len(context_set_train.union(context_set_dev).union(context_set_test)))
# print(len(context_set_train), len(context_set_dev), len(context_set_test))

# para_id_to_context_dict_train= get_para_id_to_context_dict(train)
# para_id_to_context_dict_dev= get_para_id_to_context_dict(dev)
# para_id_to_context_dict_test= get_para_id_to_context_dict(test)

TypeError: string indices must be integers

# Fix and preprocess the data

In [343]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
model = AutoModel.from_pretrained("BAAI/bge-base-en-v1.5")
model.cuda()

# Compute token embeddings
def get_sentence_embeddings(model, input_list):
    input = tokenizer(input_list, padding=True, truncation=True, return_tensors='pt')
    
    model.eval()
    with torch.no_grad():
        input = {k: v.cuda() for k, v in input.items()}
        output = model(**input)
        # Perform pooling. In this case, cls pooling.
        sentence_embeddings = output[0][:, 0]
        # print(sentence_embeddings.shape)
    # normalize embeddings
    return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)

def get_cosine_similarity(a, b, normalize=False):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
    
    return torch.matmul(a, b.T)

def get_majority_index(input_similarity_tensor):
    # Assuming that the first dimension is the context list
    input_list = input_similarity_tensor.argmax(dim=0).cpu().numpy().tolist()
    # print(input_list)
    return max(set(input_list), key = input_list.count)

def resolve_multiple_context_same_para_id(model, context_list, question_answer_list, raw_ocr):
    context_embeddings = get_sentence_embeddings(model, context_list)
    qa_embeddings = get_sentence_embeddings(model, question_answer_list)
    raw_ocr_embeddings = get_sentence_embeddings(model, [raw_ocr])
    majority_index = get_majority_index(get_cosine_similarity(context_embeddings, qa_embeddings, normalize=False))
    return majority_index, get_cosine_similarity(context_embeddings, raw_ocr_embeddings, normalize=False)


In [351]:
all = train + dev + test
all = get_para_id_to_context_dict(all)

"""
There are two cases:
1. Have exactly one context: normal
2. Have more than one context, then either:
    - Two similar paragraphs with slight difference due to punctuation or something else -> merge them into one
    - Hallucinated chatgpt context that is completely different from the raw ocr text. -> drop all these questions and contexts
"""

for k, v in tqdm(all.items()):
    if len(v['context']) > 1:
        
        context_list = v['context']
        # question_answer_list = v['question_answer']
        raw_ocr = v['raw_ocr']
        
        context_embeddings = get_sentence_embeddings(model, context_list)
        raw_ocr_embeddings = get_sentence_embeddings(model, [raw_ocr])
        
        similarity = get_cosine_similarity(context_embeddings, raw_ocr_embeddings)
        
        all[k] = {
            'context': v['context'][similarity.argmax(dim=0).item()],
            'publication_date': v['publication_date']
        }
        
        
        # majority_index, raw_ocr_similarty = resolve_multiple_context_same_para_id(model, context_list, question_answer_list, raw_ocr)
        # split_dict[k] = v['context'][majority_index]
        
        # if raw_ocr_similarty[majority_index][0] <= 0.8:
        #     print(question_answer_list)
        #     for context in context_list:
        #         print(context)
        #     print(" ".join(raw_ocr.strip().split("\n")))
        #     print(extra_output)
        #     break
    else:
        all[k] = {'context': v['context'][0], 'publication_date': v['publication_date']}

  0%|                                                                                                                                                                            | 0/131921 [00:00<?, ?it/s]/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 131921/131921 [00:02<00:00, 54650.26it/s]


In [355]:
new_train = []
for i in train:
    if all[i['para_id']]['context'] == i['context']:
        new_train.append(i)
print(len(new_train), len(train))

new_dev = []
for i in dev:
    if all[i['para_id']]['context'] == i['context']:
        new_dev.append(i)
print(len(new_dev), len(dev))

new_test = []
for i in test:
    if all[i['para_id']]['context'] == i['context']:
        new_test.append(i)
print(len(new_test), len(test))

438683 439302
24054 24111
24048 24084


# Save the corpus and filtered train/dev/test

In [356]:
all_list = []
for ix, (k, v) in enumerate(all.items()):
    all_list.append({
        "docid": k,
        "text": v['context'],
        "publication_date": v['publication_date']
    })
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/ChroniclingAmericaQA/processed/corpus.jsonl", all_list, jsonl=True)

The file contains 131921 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/ChroniclingAmericaQA/processed/corpus.jsonl


In [ ]:
train_jsonl = []
for i in new_train:
    train_jsonl.append(
        {
            "query_id": i["query_id"],
            "query": i["question"],
            "positive_passages": [{
                "docid": i["para_id"],
                "text": i["context"]
            }],
            "negative_passages": []
        }
    )
    

{'query_id': 'train_1',
 'question': 'Who is the author of the book, "Horrors of Slavery, or the American Turf in Tripoli"?',
 'answer': 'WILLIAM RAY',
 'org_answer': 'WILLIAM RAY',
 'para_id': 'New_Hampshire_18070804_1',
 'context': "Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of Slavery, or the American Turf in Tripoli,” VOTARIES of Freedom, arm! The British Lion roars! Legions of Valor, take th’ alarm—; Rash, rush to guard our shores! Behold the horrid deed— Your brethren gasping lie! Beneath a tyrant’s hand they bleed— They groan—they faint—they die. Veterans of seventy-six, Awake the slumbering sword;— Hearts of your murderous foes transfix— 'Tis vengeance gives the word. Remember Lexington, And Bunker’s tragic hill; “The same who spilt your blood thereon, Your blood again would spill. Ye who have seen your wives, Your children, an

# Create the qrel

In [6]:
test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/ChroniclingAmericaQA/processed/test.jsonl", jsonl=True)

The file is of type: <class 'list'>
The file contains 24048 items.


In [9]:
with open("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/ChroniclingAmericaQA/processed/qrel.txt", 'w') as outfile:
    for ix, v in enumerate(test_jsonl):
        query_id = v['query_id']
        docid = v['positive_passages'][0]['docid']

        outfile.write(f"{query_id} 0 {docid} {1}\n")